In [ ]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load the English Sarcasm Detector model
model_name = "helinivan/english-sarcasm-detector"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Function to preprocess text
def preprocess_data(text):
    """Lowercase and remove punctuation from the text."""
    return text.lower().strip()

# Function to classify text
def classify_sarcasm(text, tokenizer, model):
    """Classify whether a given text is sarcastic or not."""
    inputs = tokenizer(
        [preprocess_data(text)],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1).tolist()[0]
    confidence = max(probs)
    label = probs.index(confidence)
    return {"label": label, "confidence": confidence}

# Load the dataset
file_path = "/content/Bart_Iac_data_predictions.csv"  # Path to the CSV file
data = pd.read_csv(file_path)

# Check the column name (assumes the column to analyze is named 'non_sarcastic_text')
column_to_analyze = "generated"
if column_to_analyze not in data.columns:
    raise ValueError(f"Column '{column_to_analyze}' not found in the dataset!")

data = data.reset_index(drop=True)

texts = data[column_to_analyze].dropna().tolist()  # Ensure no NaN values
results = [classify_sarcasm(text, tokenizer, model) for text in texts]

data = data[data[column_to_analyze].notna()]

# Count non-sarcastic sentences
non_sarcastic_count = sum(1 for result in results if result["label"] == 0)
total_sentences = len(texts)

# Calculate accuracy
accuracy = (non_sarcastic_count / total_sentences) * 100

# Output results
print(f"Total Sentences: {total_sentences}")
print(f"Non-Sarcastic Sentences: {non_sarcastic_count}")
print(f"Accuracy (Percentage of Non-Sarcastic Sentences): {accuracy:.2f}%")

# Optional: Save results with predictions
data["sarcasm_label"] = ["Non-Sarcastic" if res["label"] == 0 else "Sarcastic" for res in results]
data["confidence"] = [res["confidence"] for res in results]
data.to_csv("sarcasm_detection_results_iacbart.csv", index=False)
print("Results saved to 'sarcasm_detection_results_iacbart.csv'.")
